# Suite PRV — Provenance: the records of the transactions a store executes

A service needs to know who changed what, when and from where ([CAP-134: transactions recorded with provenance](../../../mbse-basis/docs/capabilities/CAP-134.md);
FRAMEWORK.md, Resolved, Provenance). `Provenance`, a module installed before use (`store.transaction(Provenance.install)`),
holds a record of each executed transaction (`Provenance.OfTransaction`, keyed by its `uid`: its `when` and `where`), the
session it was executed in (`Provenance.OfSession`, keyed by its `id`, through `Provenance.InSession`) and the session's
identity (`Provenance.OfIdentity`, keyed by its `name`, through `Provenance.Who`). `Provenance.extend(txn, schemas)`
gives the reference object schemas an app chooses anonymous adjacencies to `Provenance.CreatedBy` and
`Provenance.ModifiedBy`; the executor writes the records at commit, within the transaction, from what its expression
gives (`Mutations.OfTransaction`'s `when`, `session` and `where`), so that a replay writes nothing twice. An imperative
scope's record has placeholder data (Decided, 3; tech debt).

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import json
import re

from mbse.Schemas.Framework import JSON, Builders as B, Plain, Provenance, References
from support import raises


def work(txn):
    """work.Feature, owned by the singleton board by key; work.Comment, owned by the board, which gives each its ID, and
    about a feature."""
    S = txn.Schemas
    return txn.uid("work/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Board": S.OfObject, "Holds": S.OfRelation, "Posted": S.OfRelation, "Feature": S.OfObject,
                  "Comment": S.OfObject, "About": S.OfRelation})
        .spec([
            lambda _: _.Holds.name("work.Holds").links([lambda t: t.name("board"), lambda t: t.name("feature")])
            .properties({"key": S.OfStr}).uniques([["board", "key"], ["feature"]]),
            lambda _: _.Posted.name("work.Posted").links([lambda t: t.name("board"), lambda t: t.name("comment")])
            .properties([lambda t: t.name("id").schema([S.OfStr]).pattern(r"C-(\d+)")]).uniques([["board", "id"],
                                                                                                  ["comment"]]),
            lambda _: _.Board.name("work.Board").ref(True).singleton("work.TheBoard").adjacencies({
                "features": lambda a: a.relation([_.Holds]).me("board"),
                "comments": lambda a: a.relation([_.Posted]).me("board")}),
            lambda _: _.Feature.name("work.Feature").ref(True).properties({"title": S.OfStr}).adjacencies([
                lambda t: t.name("board").relation([_.Holds]).me("feature").owner(True)]),
            lambda _: _.About.name("work.About").links([lambda t: t.name("comment"), lambda t: t.name("feature")]),
            lambda _: _.Comment.name("work.Comment").ref(True).properties({"text": S.OfStr}).adjacencies([
                lambda t: t.name("board").relation([_.Posted]).me("comment").owner(True),
                lambda t: t.name("about").relation([_.About]).me("comment")]),
        ])]))])


def made():
    """A store holding work's schemas and Provenance, work.Comment extended, and the feature `chips`."""
    store = B.Store()
    store.transaction(work)
    store.transaction(Provenance.install)
    store.transaction(lambda t: Provenance.extend(t, [t.work.Comment]))
    store.transaction(lambda t: t.uid("feature-1").when("2026-10-26T14:00:00Z").mutations([lambda mu: mu.upsert(
        lambda u: u.exists([lambda p: p.symbols({"b": store.work.Board, "f": store.work.Feature}).spec([
            lambda _: _.b.features({"chips": lambda e: e.feature(_.f)}), lambda _: _.f.title("Chips")])]))]))
    return store


def post(store, uid, when, text, session=None, where=None):
    """The expression of a transaction posting a comment about `chips`, from a session, as a service sets it."""
    def build(t):
        t.uid(uid).when(when)
        if session is not None:
            t.session(session)
        if where is not None:
            t.where(where)
        return t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
            .symbols({"topic": References.of(store, entry(store.singleton("work.TheBoard"), "features", "key", "chips").feature),
                      "c": store.work.Comment})
            .spec([lambda _: _.c.text(text).about([_.topic])])]))])
    return store.Mutations.OfTransaction(build)


def edit(store, comment, uid, text):
    """The expression of a transaction writing a comment's text, given its reference."""
    return store.Mutations.OfTransaction(lambda t: t.uid(uid).when("2026-10-26T15:00:00Z")
        .session(lambda s: s.id("tab-7f3a").who("ada")).mutations([lambda mu: mu.upsert(lambda u: u.exists([
        lambda p: p.symbols({"c": References.of(store, comment)}).spec([lambda _: _.c.text(text)])]))]))


def uids(store):
    """The uids of the records the store holds, in the order they were made."""
    return [str(entry.uid) for entry in store.singleton("Provenance.Transactions").transactions]


def record(store, uid):
    return next(entry.transaction for entry in store.singleton("Provenance.Transactions").transactions
                if str(entry.uid) == uid)


def by(store, adjacency, target):
    """The uids of the records whose adjacency `created` or `modified` links `target`."""
    return [str(entry.uid) for entry in store.singleton("Provenance.Transactions").transactions
            if any(held.object is target for held in getattr(entry.transaction, adjacency))]


def entry(singleton, adjacency, key, value):
    """A directory singleton's entry keyed by `value`."""
    return next(held for held in getattr(singleton, adjacency) if str(getattr(held, key)) == value)


def keys(singleton, adjacency, key):
    return [str(getattr(entry, key)) for entry in getattr(singleton, adjacency)]


store = made()
board = store.singleton("work.TheBoard")
chips = entry(board, "features", "key", "chips").feature

## PRV-01 · Provenance.install: its schemas and directories, run again writing nothing; installing it, and a store without it, record nothing

In [ ]:
names = ["Provenance.OfWhere", "Provenance.InSessions", "Provenance.SessionDirectory", "Provenance.OfSession",
         "Provenance.InSession", "Provenance.InTransactions", "Provenance.TransactionDirectory",
         "Provenance.OfTransaction", "Provenance.InIdentities", "Provenance.IdentityDirectory", "Provenance.OfIdentity",
         "Provenance.Who", "Provenance.CreatedBy", "Provenance.ModifiedBy"]
held = {name: store.registered(name) for name in names}
OfTransaction, OfSession = held["Provenance.OfTransaction"], held["Provenance.OfSession"]
assert list(OfTransaction.properties) == ["when", "where"] and OfTransaction.properties["where"].schema is held["Provenance.OfWhere"]
assert list(OfTransaction.adjacencies) == ["directory", "session", "created", "modified"]
assert list(OfSession.properties) == ["started", "locale", "device"] and list(OfSession.adjacencies) == [
    "directory", "transactions", "who"]
assert list(held["Provenance.OfIdentity"].adjacencies) == ["directory", "sessions"]
assert list(held["Provenance.OfWhere"].properties) == ["latitude", "longitude"] and not held["Provenance.OfWhere"].ref
for singleton in ("Provenance.Transactions", "Provenance.Sessions", "Provenance.Identities"):
    assert store.singleton(singleton) is not None
for relation, unique in (("Provenance.InSession", "session"), ("Provenance.Who", "identity"),
                         ("Provenance.CreatedBy", "transaction"), ("Provenance.ModifiedBy", "transaction")):
    assert held[relation].uniques == (frozenset({unique}),)  # the other link determines it: one per transaction, ...
assert uids(store) == ["feature-1"]  # neither the install nor the extension, which write schemas alone
before = {name: store.registered(name) for name in store.names()}
store.transaction(Provenance.install)  # each found by its name
assert {name: store.registered(name) for name in store.names()} == before and uids(store) == ["feature-1"]
plain = B.Store()  # no Provenance: nothing recorded, nothing changes
plain.transaction(work)
plain.transaction(lambda t: t.uid("feature-1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"b": plain.work.Board, "f": plain.work.Feature}).spec([
        lambda _: _.b.features({"chips": lambda e: e.feature(_.f)}), lambda _: _.f.title("Chips")])]))]))
assert not any(name.startswith("Provenance.") for name in plain.names())
assert str(entry(plain.singleton("work.TheBoard"), "features", "key", "chips").feature.title) == "Chips"

## PRV-02 · Provenance.extend: anonymous adjacencies to CreatedBy and ModifiedBy through `object`, no member; run again, writing nothing; refused before Provenance is installed, or for what is not a reference object schema

In [ ]:
Comment, Feature = store.registered("work.Comment"), store.registered("work.Feature")
added = [(a.relation.name, a.me, a.name) for a in Comment.anonymous]
assert added == [("Provenance.CreatedBy", "object", ""), ("Provenance.ModifiedBy", "object", "")]
assert list(Comment.adjacencies) == ["board", "about"] and Feature.anonymous == []  # no member; Feature not extended
store.transaction(lambda t: Provenance.extend(t, [Comment]))  # a schema object, or its path: found, nothing written
assert [(a.relation.name, a.me) for a in Comment.anonymous] == [(r, m) for r, m, _ in added]
bare = B.Store()
bare.transaction(work)
with raises(LookupError, match="Provenance.extend: the store has no Provenance installed; install it first, "
            "store.transaction(Provenance.install)"):
    bare.transaction(lambda t: Provenance.extend(t, [t.work.Comment]))
for wrong, shown in ((lambda t: t.Provenance.OfWhere, "'Provenance.OfWhere'"), (lambda t: "work.Comment", "'work.Comment'")):
    with raises(TypeError, match=f"Provenance.extend takes reference object schemas: {shown} is not one"):
        store.transaction(lambda t: Provenance.extend(t, [wrong(t)]))

## PRV-03 · A transaction with `when`, `session` and `where`: its record, its session and identity made, InSession, Who, and CreatedBy for the comment it creates

In [ ]:
first = post(store, "6f1c4e2a-0000-4000-8000-000000000007", "2026-10-26T14:30:05Z", "Should chips show adjacencies?",
             lambda s: s.id("tab-7f3a").who("ada").locale("en-GB"), lambda w: w.latitude(51.5).longitude(-0.125))
comment = store.transaction(first)[0]["c"]
assert uids(store) == ["feature-1", "6f1c4e2a-0000-4000-8000-000000000007"]
made_ = record(store, "6f1c4e2a-0000-4000-8000-000000000007")
assert str(made_.when) == "2026-10-26T14:30:05Z"
where = store.member(made_, "where")
assert (store.member(where, "latitude"), store.member(where, "longitude")) == (51.5, -0.125)
sessions = store.singleton("Provenance.Sessions")
assert keys(sessions, "sessions", "id") == ["unknown", "tab-7f3a"]  # feature-1's, which named none; tab-7f3a
session = entry(sessions, "sessions", "id", "tab-7f3a").session
assert (str(session.started), str(session.locale), store.member(session, "device")) == ("2026-10-26T14:30:05Z", "en-GB", None)
assert [entry.session for entry in made_.session] == [session]  # InSession
identities = store.singleton("Provenance.Identities")
assert keys(identities, "identities", "name") == ["unknown", "ada"]  # feature-1's, which named no session; ada's
ada = entry(identities, "identities", "name", "ada").identity
assert [entry.identity for entry in session.who] == [ada] and [entry.session for entry in ada.sessions] == [session]
assert by(store, "created", comment) == ["6f1c4e2a-0000-4000-8000-000000000007"] and by(store, "modified", comment) == []
assert str(entry(board, "comments", "id", "C-1").comment.text) == "Should chips show adjacencies?"  # its directory gave it its ID

## PRV-04 · A later write gives ModifiedBy, replaced by the next: the tail; the session found by its id, another session of the identity found by its name

In [ ]:
store.transaction(edit(store, comment, "edit-2", "Should chips show their adjacencies?"))
assert by(store, "created", comment) == ["6f1c4e2a-0000-4000-8000-000000000007"] and by(store, "modified", comment) == ["edit-2"]
store.transaction(edit(store, comment, "edit-3", "Chips should show their adjacencies."))
assert by(store, "modified", comment) == ["edit-3"] and uids(store)[-2:] == ["edit-2", "edit-3"]  # edit-2's record stays
assert keys(sessions, "sessions", "id") == ["unknown", "tab-7f3a"] and len(list(session.transactions)) == 3  # found by its id
assert str(record(store, "edit-2").when) == "2026-10-26T15:00:00Z" and store.member(record(store, "edit-2"), "where") is None
store.transaction(edit(store, comment, "edit-3", "Again."))  # another write under the same uid
assert by(store, "modified", comment) == ["edit-3"] and uids(store).count("edit-3") == 1  # its uid's record, found
phone = post(store, "phone-1", "2026-10-26T16:00:00Z", "From my phone.", lambda s: s.id("tab-9b2c").who("ada").device(
    "phone"))
store.transaction(phone)
assert keys(sessions, "sessions", "id") == ["unknown", "tab-7f3a", "tab-9b2c"] and keys(identities, "identities", "name") == [
    "unknown", "ada"]
other = entry(sessions, "sessions", "id", "tab-9b2c").session
assert (store.member(other, "locale"), str(other.device)) == (None, "phone") and [e.identity for e in other.who] == [ada]

## PRV-05 · Objects of schemas not extended are not linked; a transaction that writes nothing records nothing; one that creates and writes an object in it, CreatedBy alone

In [ ]:
store.transaction(lambda t: t.uid("retitle").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"f": References.of(store, chips)}).spec([lambda _: _.f.title("Chips, again")])]))]))
assert "retitle" in uids(store) and list(record(store, "retitle").created) == list(record(store, "retitle").modified) == []
count = len(uids(store))
assert store.transaction(lambda t: t.uid("read").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"c": store.work.Comment})]))])) != []  # a map: nothing written
assert len(uids(store)) == count
both = store.transaction(lambda t: t.uid("both").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"c": store.work.Comment}).spec([lambda _: _.c.text("new")])]).then([lambda n: n.update(lambda d: d.apply([
    lambda p: p.symbols({"c": store.work.Comment}).spec([lambda _: _.c.text("newer")])]))]))]))[0]["c"]
assert by(store, "created", both) == ["both"] and by(store, "modified", both) == []

## PRV-06 · Replaying the same expression writes nothing twice; replayed into another store, the journal makes the same records

In [ ]:
journal = [json.dumps(Plain.ToPlain(store).Transaction(expression))  # the expressions, as a journal holds them
           for expression in (first, edit(store, comment, "edit-2", "Should chips show their adjacencies?"))]
again = made()
copy = again.transaction(Plain.FromPlain(again).Transaction(json.loads(journal[0])))[0]["c"]
directories = [again.singleton(name) for name in ("Provenance.Transactions", "Provenance.Sessions", "Provenance.Identities",
                                                  "work.TheBoard")]
sizes = lambda: [len(list(getattr(d, a))) for d, a in zip(directories, ("transactions", "sessions", "identities",
                                                                        "comments"))] + [len(list(copy.about))]
before = sizes()
assert again.transaction(Plain.FromPlain(again).Transaction(json.loads(journal[0])))[0]["c"] is copy  # replayed: found again
assert sizes() == before and by(again, "created", copy) == ["6f1c4e2a-0000-4000-8000-000000000007"]  # nothing twice
again.transaction(Plain.FromPlain(again).Transaction(json.loads(journal[1])))
assert by(again, "created", copy) == ["6f1c4e2a-0000-4000-8000-000000000007"] and by(again, "modified", copy) == ["edit-2"]
assert str(record(again, "6f1c4e2a-0000-4000-8000-000000000007").when) == "2026-10-26T14:30:05Z"  # the same records
assert keys(again.singleton("Provenance.Sessions"), "sessions", "id") == ["unknown", "tab-7f3a"]
assert str(entry(again.singleton("Provenance.Sessions"), "sessions", "id", "tab-7f3a").session.started) == \
    "2026-10-26T14:30:05Z"

## PRV-07 · The new transaction members on the wire: `when`, `session` and `where`, written and read back; left out where unused

In [ ]:
written = Plain.ToPlain(store).Transaction(first)
assert {k: written[k] for k in ("uid", "when", "session", "where")} == {
    "uid": "6f1c4e2a-0000-4000-8000-000000000007", "when": "2026-10-26T14:30:05Z",
    "session": {"id": "tab-7f3a", "who": "ada", "locale": "en-GB"}, "where": {"latitude": 51.5, "longitude": -0.125}}
assert Plain.ToPlain(store).Transaction(Plain.FromPlain(store).Transaction(written)) == written
text = json.dumps(written)  # as JSON
assert Plain.ToPlain(store).Transaction(Plain.FromPlain(store).Transaction(json.loads(text))) == written
bare = store.Mutations.OfTransaction(lambda t: t.uid("x"))
assert Plain.ToPlain(store).Transaction(bare) == {"uid": "x"}  # none of them: as before
Session = store.registered("Mutations.OfTransaction.Session")
assert list(Session.properties) == ["id", "who", "locale", "device"] and not Session.ref
assert list(store.registered("Mutations.OfTransaction.Where").properties) == ["latitude", "longitude"]
assert list(store.registered("Mutations.OfTransaction").properties) == ["uid", "when", "session", "where"]

## PRV-08 · An imperative scope, and an expression naming no session or time: placeholder data, a uid the store makes, `when` at commit, the session and identity `unknown`

In [ ]:
with store.transaction():
    loose = store.work.Comment(lambda c: c.text("by hand").about([chips]))
    store.work.Comment(loose, lambda c: c.text("by hand, edited"))  # created and written in it: CreatedBy alone
(uid,) = by(store, "created", loose)
assert re.fullmatch(r"[0-9a-f]{8}-[0-9a-f]{4}-4[0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}", uid) and by(store, "modified", loose) == []
placeholder = record(store, uid)
assert re.fullmatch(r"\d{4}-\d\d-\d\dT\d\d:\d\d:\d\dZ", str(placeholder.when))
(held,) = placeholder.session
assert [str(entry.id) for entry in held.session.directory] == ["unknown"]
assert [str(e.name) for e in next(iter(held.session.who)).identity.directory] == ["unknown"]
store.transaction(edit(store, loose, "edit-loose", "by a service"))
assert by(store, "modified", loose) == ["edit-loose"]
store.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols({
    "c": References.of(store, loose)}).spec([lambda _: _.c.text("no uid, no when")])]))]))
(latest,) = by(store, "modified", loose)
assert latest not in ("edit-loose", uid) and re.fullmatch(r"\d{4}-.*Z", str(record(store, latest).when))
assert [str(e.id) for e in next(iter(record(store, latest).session)).session.directory] == ["unknown"]

## PRV-09 · Reading an object's records from the record side: a map from a symbol given the object's reference, the record, its session and identity

In [ ]:
frames = store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"c": References.of(store, comment), "t": store.Provenance.OfTransaction, "i": store.Provenance.OfIdentity,
              "s": store.Provenance.OfSession})
    .spec([lambda _: _.t.created([_.c]), lambda _: _.t.session([_.s]), lambda _: _.s.who([_.i])])]))]))
assert len(frames) == 1  # one frame: t binds the record (its `when`), i the identity (its `name`), s the session
found = frames[0]
assert str(found["t"].when) == "2026-10-26T14:30:05Z" and str(found["s"].locale) == "en-GB"
assert [str(entry.name) for entry in found["i"].directory] == ["ada"]
latest = store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"c": References.of(store, comment), "t": store.Provenance.OfTransaction})
    .spec([lambda _: _.t.modified([_.c])])]))]))
assert [[str(entry.uid) for entry in frame["t"].directory] for frame in latest] == [["edit-3"]]  # the tail

## PRV-10 · What is refused, the transaction with it: a session with no id; a session of another identity

In [ ]:
count = len(uids(store))
nameless = post(store, "nameless", "2026-10-26T17:00:00Z", "No id.", lambda s: s.who("ada"))
with raises(ValueError, match="a transaction's session has no id: a service gives it, and who, before executing it"):
    store.transaction(nameless)
stolen = post(store, "stolen", "2026-10-26T17:00:00Z", "Not mine.", lambda s: s.id("tab-7f3a").who("eve"))
with raises(ValueError, match="session 'tab-7f3a' is another identity's, not 'eve''s: a session has one identity "
            "(Provenance.Who)"):
    store.transaction(stolen)
assert len(uids(store)) == count and "Not mine." not in [str(e.comment.text) for e in board.comments]  # nothing written
assert keys(identities, "identities", "name") == ["unknown", "ada"]

## PRV-11 · Reading a store's snapshot back restores its records and records nothing of its own, read alone or within a transaction: a restart adds no record

In [ ]:
source = made()
source.transaction(post(source, "post-1", "2026-10-26T15:00:00Z", "Restart me", lambda s: s.id("tab-9").who("ada")))
text = JSON.ToJSON(source).Snapshot()  # as a restart writes it, and reads it
for restoring in ("alone", "within"):
    restored = B.Store()
    restored.transaction(work)
    restored.transaction(Provenance.install)
    if restoring == "alone":
        JSON.FromJSON(restored).Snapshot(text)
    else:
        with restored.transaction():  # within a transaction open: its writes are the snapshot's, not its own
            JSON.FromJSON(restored).Snapshot(text)
    twin = next(held.comment for held in restored.singleton("work.TheBoard").comments
                if str(held.comment.text) == "Restart me")
    assert by(restored, "created", twin) == ["post-1"]  # one creating transaction, restored
    assert uids(restored) == uids(source)  # the records restored, none added
    assert JSON.ToJSON(restored).Snapshot() == text  # restarted again: the same snapshot